# CS60010 — Deep Learning | Spring 2026
## Assignment 2: TRM Ablation Study (3 Variants)
---
**Author:**
Arghyadeep Ghosh (25CS91R03) - IIT Kharagpur
Pallab Biswas (23EE10092) · IIT Kharagpur

Three targeted ablations of the Tiny Recursive Model (TRM):

| # | Variant | What changes |
|---|---------|--------------|
| ABL-1 | **2-D RoPE** | Replace 3-D RoPE (row, col, pair) with 2-D RoPE (row, col only) |
| ABL-2 | **CE-only loss** | Replace deep-supervision loss with final-step CE loss only |
| ABL-3 | **Minimal cycles** | Reduce reasoning depth: 1 inner × 1 outer cycle (vs 6 × 3) |



## Imports & global config

In [ ]:
import os, json, math, copy, random, itertools
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from einops import rearrange

# Reproducibility 
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Device 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Using device: {DEVICE}")

# ARC constants 
NUM_COLORS   = 10
MAX_GRID_DIM = 30
MAX_SEQ_LEN  = 2048
PAD_TOKEN    = 10
SEP_TOKEN    = 11
NUM_TOKENS   = 12

# Paths 
DATA_ROOT = Path("/home/arghyadeep/ARC/data")
TRAIN_DIR = DATA_ROOT / "training"
EVAL_DIR  = DATA_ROOT / "evaluation"
CKPT_DIR  = Path("/home/arghyadeep/ARC/checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)


# ── Dihedral-8 augmentations ──────────────────────────────────────────────────
def rotate90(g): return np.rot90(g, k=1).copy()
def flip_h(g):   return np.fliplr(g).copy()

DIHEDRAL_OPS = [
    lambda g: g,
    lambda g: rotate90(g),
    lambda g: rotate90(rotate90(g)),
    lambda g: rotate90(rotate90(rotate90(g))),
    lambda g: flip_h(g),
    lambda g: flip_h(rotate90(g)),
    lambda g: flip_h(rotate90(rotate90(g))),
    lambda g: flip_h(rotate90(rotate90(rotate90(g)))),
]



[Config] Using device: cuda


## Dataset loading & augmentation (unchanged from original)

In [5]:
def load_arc_tasks(task_dir: Path) -> List[Dict]:
    tasks = []
    for fp in sorted(task_dir.glob("*.json")):
        with open(fp) as f:
            raw = json.load(f)
        tasks.append({"task_id": fp.stem, "train": raw["train"], "test": raw["test"]})
    return tasks

def augment_geometric(task, op_idx):
    op = DIHEDRAL_OPS[op_idx]
    def ap(pair, with_out=True):
        inp = op(np.array(pair["input"], dtype=np.int8))
        if with_out and "output" in pair:
            return {"input": inp.tolist(), "output": op(np.array(pair["output"], dtype=np.int8)).tolist()}
        return {"input": inp.tolist()}
    return {"task_id": task["task_id"]+f"_d{op_idx}",
            "train":   [ap(p) for p in task["train"]],
            "test":    [ap(p, False) for p in task["test"]]}

def is_colour_identity(task):
    for p in task["train"]:
        if set(np.array(p["output"]).flatten()) - set(np.array(p["input"]).flatten()):
            return True
    return False

def augment_colour(task, perm):
    lut = np.array(perm, dtype=np.int8)
    def ap(g): return lut[np.array(g, dtype=np.int8)].tolist()
    def apair(p, with_out=True):
        d = {"input": ap(p["input"])}
        if with_out and "output" in p: d["output"] = ap(p["output"])
        return d
    return {"task_id": task["task_id"]+"_cp",
            "train": [apair(p) for p in task["train"]],
            "test":  [apair(p, False) for p in task["test"]]}

def build_augmented_dataset(raw_tasks, colour_aug=True):
    augmented = []
    for task in raw_tasks:
        for i in range(8):
            augmented.append(augment_geometric(task, i))
        if colour_aug and not is_colour_identity(task):
            perm = list(range(10)); random.shuffle(perm)
            augmented.append(augment_colour(task, perm))
    print(f"[Dataset] {len(raw_tasks)} tasks → {len(augmented)} augmented")
    return augmented

def grid_to_tokens(grid):
    return torch.tensor([c for row in grid for c in row], dtype=torch.long)

def tokens_to_grid(tokens, H, W):
    flat = tokens.cpu().numpy().tolist()[:H*W]
    return [[min(9, max(0, flat[r*W+c])) for c in range(W)] for r in range(H)]

def build_sequence(task, pair_idx_for_test=0):
    demo_pairs = task["train"]
    test_pair  = task["test"][pair_idx_for_test]
    src, pidx, rcids = [], [], []
    for p, pair in enumerate(demo_pairs):
        for r, row in enumerate(pair["input"]):
            for c, val in enumerate(row):
                src.append(val); pidx.append(p); rcids.append((r, c))
        src.append(SEP_TOKEN); pidx.append(p); rcids.append((0, 0))
        for r, row in enumerate(pair["output"]):
            for c, val in enumerate(row):
                src.append(val); pidx.append(p); rcids.append((r, c))
        src.append(SEP_TOKEN); pidx.append(p); rcids.append((0, 0))
    test_inp = test_pair["input"]
    tgt_grid = test_pair.get("output", None)
    tgt_h    = len(tgt_grid)    if tgt_grid else 0
    tgt_w    = len(tgt_grid[0]) if tgt_grid else 0
    n_demo   = len(demo_pairs)
    for r, row in enumerate(test_inp):
        for c, val in enumerate(row):
            src.append(val); pidx.append(n_demo); rcids.append((r, c))
    src = src[:MAX_SEQ_LEN]; pidx = pidx[:MAX_SEQ_LEN]; rcids = rcids[:MAX_SEQ_LEN]
    tgt = grid_to_tokens(tgt_grid) if tgt_grid else torch.zeros(1, dtype=torch.long)
    return (torch.tensor(src, dtype=torch.long), tgt,
            torch.tensor(pidx, dtype=torch.long), rcids, tgt_h, tgt_w)

class ARCDataset(Dataset):
    def __init__(self, tasks):
        self.samples = []
        for t in tasks:
            n = len(t["train"])
            if n < 2: continue
            for hold in range(n):
                self.samples.append((t, hold))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        t, hold = self.samples[idx]
        ctx_task = {
            "task_id": t["task_id"],
            "train":   [p for i, p in enumerate(t["train"]) if i != hold],
            "test":    [t["train"][hold]],
        }
        src, tgt, pidx, rcids, th, tw = build_sequence(ctx_task, pair_idx_for_test=0)
        return {"src": src, "tgt": tgt, "pair_idx": pidx,
                "rc_ids": rcids, "tgt_h": th, "tgt_w": tw, "task_id": t["task_id"]}

def arc_collate_fn(batch):
    ml = max(b["src"].shape[0] for b in batch)
    sp, pp, ms, rc = [], [], [], []
    for b in batch:
        L = b["src"].shape[0]; pad = ml - L
        sp.append(F.pad(b["src"], (0, pad), value=PAD_TOKEN))
        pp.append(F.pad(b["pair_idx"], (0, pad), value=0))
        m = torch.zeros(ml, dtype=torch.bool); m[L:] = True; ms.append(m)
        rc.append(b["rc_ids"] + [(0, 0)] * pad)
    return {"src": torch.stack(sp), "pair_idx": torch.stack(pp),
            "src_mask": torch.stack(ms), "rc_ids": rc,
            "tgt": [b["tgt"] for b in batch],
            "tgt_h": [b["tgt_h"] for b in batch],
            "tgt_w": [b["tgt_w"] for b in batch],
            "task_id": [b["task_id"] for b in batch]}

print(" Dataset utilities loaded.")


 Dataset utilities loaded.


## Positional Encodings

### ABL-1: 2-D RoPE vs original 3-D RoPE
- **3-D RoPE** (original): encodes `(row, col, pair_index)` — head_dim split into 3 equal parts
- **2-D RoPE** (ablation): encodes `(row, col)` only — pair information dropped, head_dim split into 2 parts

The `pair_index` dimension in 3-D RoPE allows the model to distinguish which demo pair a token
belongs to. Without it the model must rely solely on token-embedding pair offsets, which is a
weaker inductive bias for relational reasoning.


In [6]:
# Original: 3-D RoPE (row, col, pair_index)
class RoPE3D(nn.Module):
    """
    Three-dimensional RoPE (row, col, pair_index).
    head_dim must be divisible by 6.
    Used by: TRM (original), ABL-2, ABL-3.
    """
    def __init__(self, head_dim: int, max_pos: int = 64, max_pairs: int = 8):
        super().__init__()
        assert head_dim % 6 == 0, "head_dim must be divisible by 6 for 3-D RoPE"
        self.dim_each = head_dim // 3
        for name, mx in [("row", max_pos), ("col", max_pos), ("pair", max_pairs)]:
            d = self.dim_each // 2
            inv = 1.0 / (10000 ** (torch.arange(0, d, dtype=torch.float) / d))
            self.register_buffer(f"inv_freq_{name}", inv)

    def _cs(self, pos, inv):
        f = torch.outer(pos.float(), inv)
        e = torch.cat([f, f], dim=-1)
        return e.cos(), e.sin()

    def rotate_half(self, x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, q, k, rows, cols, pairs):
        T = q.shape[2]
        cos_r, sin_r = self._cs(rows[:T], self.inv_freq_row)
        cos_c, sin_c = self._cs(cols[:T], self.inv_freq_col)
        cos_p, sin_p = self._cs(pairs[:T], self.inv_freq_pair)
        cos = torch.cat([cos_r, cos_c, cos_p], -1).to(q.device)[None, None]
        sin = torch.cat([sin_r, sin_c, sin_p], -1).to(q.device)[None, None]
        return q*cos + self.rotate_half(q)*sin, k*cos + self.rotate_half(k)*sin


# ABL-1: 2-D RoPE (row, col only — pair_index dropped)
class RoPE2D(nn.Module):
    """
    ABL-1 — Two-dimensional RoPE (row, col) only. Pair-index dropped.
    head_dim must be divisible by 4 (2 axes × 2 sin/cos components).

    Parameter delta vs 3-D RoPE:
      3-D RoPE: 3 × (head_dim//6) inv_freq buffers (no learned params)
      2-D RoPE: 2 × (head_dim//4) inv_freq buffers (no learned params)
      ΔParams = 0  (both are parameter-free)
    The only difference is that pair relationship must be captured by
    the learned pair_emb in the token embedding, not by the rotation.
    """
    def __init__(self, head_dim: int, max_pos: int = 64):
        super().__init__()
        assert head_dim % 4 == 0, "head_dim must be divisible by 4 for 2-D RoPE"
        self.dim_each = head_dim // 2   # half head_dim per axis
        for name in ["row", "col"]:
            d = self.dim_each // 2
            inv = 1.0 / (10000 ** (torch.arange(0, d, dtype=torch.float) / d))
            self.register_buffer(f"inv_freq_{name}", inv)

    def _cs(self, pos, inv):
        f = torch.outer(pos.float(), inv)
        e = torch.cat([f, f], dim=-1)
        return e.cos(), e.sin()

    def rotate_half(self, x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, q, k, rows, cols, pairs=None):
        # pairs argument kept for API compatibility — ignored in 2-D RoPE
        T = q.shape[2]
        cos_r, sin_r = self._cs(rows[:T], self.inv_freq_row)
        cos_c, sin_c = self._cs(cols[:T], self.inv_freq_col)
        cos = torch.cat([cos_r, cos_c], -1).to(q.device)[None, None]
        sin = torch.cat([sin_r, sin_c], -1).to(q.device)[None, None]
        return q*cos + self.rotate_half(q)*sin, k*cos + self.rotate_half(k)*sin


print("RoPE3D (original) and RoPE2D (ABL-1) loaded.")


RoPE3D (original) and RoPE2D (ABL-1) loaded.


## Transformer blocks (rope_cls-parameterised)

In [7]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, rope_cls=RoPE3D):
        super().__init__()
        self.n_heads = n_heads
        self.hd = d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3*d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        self.rope = rope_cls(self.hd)   # ← swappable RoPE

    def forward(self, x, rows, cols, pairs, key_padding_mask=None):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.hd).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = self.rope(q, k, rows, cols, pairs)
        attn = (q @ k.transpose(-2,-1)) / math.sqrt(self.hd)
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask[:,None,None,:], float('-inf'))
        attn = self.drop(F.softmax(attn, dim=-1))
        out  = (attn @ v).transpose(1,2).reshape(B, T, D)
        return self.proj(out)

class FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, rope_cls=RoPE3D):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadSelfAttention(d_model, n_heads, dropout, rope_cls)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = FFN(d_model, d_ff, dropout)
    def forward(self, x, rows, cols, pairs, kpm=None):
        x = x + self.attn(self.norm1(x), rows, cols, pairs, kpm)
        x = x + self.ffn(self.norm2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, n_layers=2, dropout=0.1, rope_cls=RoPE3D):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout, rope_cls)
             for _ in range(n_layers)])
    def forward(self, x, rows, cols, pairs, kpm=None):
        for l in self.layers: x = l(x, rows, cols, pairs, kpm)
        return x

print(" Transformer blocks loaded.")


 Transformer blocks loaded.


## TRM (rope_cls-parameterised, supports all 3 ablations)

In [8]:
class TRM(nn.Module):
    """
    Tiny Recursive Model — parameterised to support all 3 ablations:

      rope_cls  : RoPE3D (original) | RoPE2D (ABL-1)
      use_deep_supervision : True (original) | False (ABL-2, CE-only)
      n_inner   : 6 (original) | 1 (ABL-3)
      T_cycles  : 3 (original) | 1 (ABL-3)
    """
    def __init__(self, d_model=96, n_heads=4, d_ff=256, n_layers=2,
                 n_inner=6, T_cycles=3, dropout=0.1, max_pairs=16,
                 rope_cls=RoPE3D, use_deep_supervision=True):
        super().__init__()
        self.d_model  = d_model
        self.n_inner  = n_inner
        self.T_cycles = T_cycles
        self.use_deep_supervision = use_deep_supervision

        self.token_emb = nn.Embedding(NUM_TOKENS, d_model, padding_idx=PAD_TOKEN)
        self.pair_emb  = nn.Embedding(max_pairs+1, d_model)

        self.proj_in  = nn.Linear(3*d_model, d_model)
        self.f_inner  = TinyTransformer(d_model, n_heads, d_ff, n_layers, dropout, rope_cls)

        self.proj_ya  = nn.Linear(2*d_model, d_model)
        self.f_update = TinyTransformer(d_model, n_heads, d_ff, n_layers, dropout, rope_cls)

        self.head      = nn.Linear(d_model, NUM_COLORS)
        self.halt_head = nn.Linear(d_model, 1)
        self.drop      = nn.Dropout(dropout)

    def embed(self, src, pair_idx):
        return self.drop(self.token_emb(src) + self.pair_emb(pair_idx))

    def forward(self, src, pair_idx, tgt_len,
                rows_ctx, cols_ctx, pairs_ctx,
                rows_tgt, cols_tgt, pairs_tgt, src_mask=None):
        B = src.shape[0]
        x = self.embed(src, pair_idx)

        y = torch.zeros(B, tgt_len, self.d_model, device=src.device)
        z = torch.zeros(B, tgt_len, self.d_model, device=src.device)

        all_logits = []
        for _ in range(self.T_cycles):
            for _ in range(self.n_inner):
                ctx = x.mean(1, keepdim=True).expand(B, tgt_len, -1)
                z   = self.f_inner(
                        F.gelu(self.proj_in(torch.cat([ctx, y, z], -1))),
                        rows_tgt, cols_tgt, pairs_tgt)
            y = self.f_update(
                    F.gelu(self.proj_ya(torch.cat([y, z], -1))),
                    rows_tgt, cols_tgt, pairs_tgt)
            all_logits.append(self.head(y))

        return all_logits[-1], all_logits

print(" TRM (parameterised) loaded.")


 TRM (parameterised) loaded.


## Parameter counter, loss functions, EMA, scheduler

In [ ]:
def count_parameters(model, label="Model"):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    for name, mod in model.named_children():
        n = sum(p.numel() for p in mod.parameters() if p.requires_grad)
        print(f"  {name:<25s}  {n:>12,d}")
    print(f"{'─'*60}")
    print(f"  {'TOTAL':<25s}  {total:>12,d}")
    assert total <= 50_000_000, f"BUDGET EXCEEDED: {total:,} > 50,000,000"
    print(f"  ✓ Within 50M budget  ({total/1e6:.4f}M / 50.000M)")
    print(f"{'='*60}\n")
    return total

def build_pos_tensors(rc_ids):
    rows = torch.tensor([r for r, _ in rc_ids], dtype=torch.long)
    cols = torch.tensor([c for _, c in rc_ids], dtype=torch.long)
    return rows, cols

def tgt_pos_tensors(H, W, pair_idx=4):
    rows, cols = [], []
    for r in range(H):
        for c in range(W):
            rows.append(r); cols.append(c)
    T = len(rows)
    return (torch.tensor(rows, dtype=torch.long),
            torch.tensor(cols, dtype=torch.long),
            torch.full((T,), pair_idx, dtype=torch.long))

#  ABL-2: CE-only loss (only final-step logits used) 
def ce_only_loss(all_logits, tgt):
    """ABL-2: ignore intermediate cycle logits — only use the final answer."""
    lg = all_logits[-1]
    B, L, C = lg.shape
    return F.cross_entropy(lg.reshape(B*L, C), tgt.reshape(B*L),
                           ignore_index=PAD_TOKEN)

# Original: deep-supervision loss 
def deep_supervision_loss(all_logits, tgt, gamma=0.5):
    """Original: earlier cycles discounted by gamma^(T-t-1)."""
    T  = len(all_logits)
    ws = [gamma**(T-1-t) for t in range(T)]
    loss = torch.tensor(0.0, device=all_logits[0].device)
    for lg, w in zip(all_logits, ws):
        B, L, C = lg.shape
        loss += w * F.cross_entropy(lg.reshape(B*L, C), tgt.reshape(B*L),
                                    ignore_index=PAD_TOKEN)
    return loss / sum(ws)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {n: p.data.clone().float()
                       for n, p in model.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.decay*self.shadow[n] + (1-self.decay)*p.data.float()
    def apply(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow: p.data.copy_(self.shadow[n].to(p.device))
    def restore(self, model, state):
        model.load_state_dict(state)

def build_wsd_scheduler(optimizer, total_steps, warmup_frac=0.05, decay_frac=0.20):
    we = int(total_steps * warmup_frac)
    ds = int(total_steps * (1 - decay_frac))
    def f(s):
        if s < we:  return s / max(1, we)
        if s < ds:  return 1.0
        return max(0.0, 1.0 - (s - ds) / max(1, total_steps - ds))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)

print(" Utilities loaded.")


 Utilities loaded.


## Generic training loop (loss_fn parameter)

In [10]:
def train_one_epoch(model, loader, optimizer, scheduler, ema, scaler,
                    device, accum_steps=8, loss_fn=deep_supervision_loss):
    """
    loss_fn: deep_supervision_loss (default) | ce_only_loss (ABL-2)
    All other ablations only change the model, so loss_fn stays default.
    """
    model.train()
    running_loss = 0.0
    running_n    = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        src      = batch["src"].to(device)
        pair_idx = batch["pair_idx"].to(device)
        src_mask = batch["src_mask"].to(device)
        B = src.shape[0]
        bloss = torch.tensor(0.0, device=device)
        valid = 0

        for i in range(B):
            th, tw = batch["tgt_h"][i], batch["tgt_w"][i]
            if th == 0: continue
            tgt_i = batch["tgt"][i].to(device)
            T_out = th * tw
            rcids = batch["rc_ids"][i]
            rc, cc = build_pos_tensors(rcids)
            rt, ct, pt = tgt_pos_tensors(th, tw, 4)

            with autocast('cuda', dtype=torch.bfloat16):
                _, all_logits = model(
                    src[i:i+1], pair_idx[i:i+1], T_out,
                    rc.to(device), cc.to(device), pair_idx[i],
                    rt.to(device), ct.to(device), pt.to(device),
                    src_mask[i:i+1])
                loss = loss_fn(all_logits, tgt_i[:T_out].unsqueeze(0))

            bloss += loss
            valid += 1

        if valid == 0: continue
        running_loss += bloss.item()
        running_n    += valid
        bloss = bloss / (valid * accum_steps)
        scaler.scale(bloss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            ema.update(model)
            if scheduler is not None:
                scheduler.step()

    return running_loss / max(1, running_n)

print(" Training loop loaded.")


 Training loop loaded.


## Inference & evaluation (pixel accuracy + exact match)

In [11]:
@torch.no_grad()
def predict_task(model, task, device, pair_idx_for_test=0, attempts=2):
    model.eval()
    src, tgt, pidx, rcids, th, tw = build_sequence(task, pair_idx_for_test)
    if th == 0:
        ti = task["test"][pair_idx_for_test]["input"]
        th, tw = len(ti), len(ti[0])
    src_d  = src.unsqueeze(0).to(device)
    pidx_d = pidx.unsqueeze(0).to(device)
    pidx_d = pidx_d.clamp(0, 16)  # guard against pair_idx overflow
    T_out  = th * tw
    rc, cc = build_pos_tensors(rcids)
    rt, ct, pt = tgt_pos_tensors(th, tw, 4)
    preds = []
    for temp in [0.0, 0.5][:attempts]:
        with autocast('cuda', dtype=torch.bfloat16):
            fl, _ = model(src_d, pidx_d, T_out,
                          rc.to(device), cc.to(device), pidx_d.squeeze(0),
                          rt.to(device), ct.to(device), pt.to(device))
        lg = fl[0].float()
        if temp == 0.0:
            toks = lg.argmax(-1)
        else:
            toks = torch.multinomial(F.softmax(lg / temp, -1), 1).squeeze(-1)
        preds.append(tokens_to_grid(toks, th, tw))
    return preds

def evaluate_pixel_accuracy(model, tasks, device, max_tasks=None):
    """Pixel-wise accuracy across all tasks."""
    correct = 0; total = 0
    n = len(tasks) if max_tasks is None else min(len(tasks), max_tasks)
    model.eval()
    for task in tasks[:n]:
        for pi, tp in enumerate(task["test"]):
            if "output" not in tp: continue
            gt = tp["output"]
            pred = predict_task(model, task, device, pi, attempts=1)[0]
            th, tw = len(gt), len(gt[0])
            if len(pred) != th or len(pred[0]) != tw: continue
            for r in range(th):
                for c in range(tw):
                    if pred[r][c] == gt[r][c]: correct += 1
                    total += 1
    pix_acc = correct / max(1, total)
    return pix_acc

def evaluate_exact_match(model, tasks, device, max_tasks=None):
    solved = 0
    n = len(tasks) if max_tasks is None else min(len(tasks), max_tasks)
    for task in tasks[:n]:
        ok = True
        for pi, tp in enumerate(task["test"]):
            if "output" not in tp: ok = False; break
            if not any(p == tp["output"] for p in predict_task(model, task, device, pi)):
                ok = False; break
        if ok: solved += 1
    acc = solved / n
    return solved, n, acc

print(" Evaluation utilities loaded.")



 Evaluation utilities loaded.


##  Instantiate original TRM & count parameters

In [ ]:
# ── Original TRM (already trained for 40 epochs) ─────────────────────────────
original_trm = TRM(
    d_model=96, n_heads=4, d_ff=256, n_layers=2,
    n_inner=6, T_cycles=3, dropout=0.1, max_pairs=16,
    rope_cls=RoPE3D, use_deep_supervision=True
).to(DEVICE)

orig_params = count_parameters(original_trm, "Original TRM (3-D RoPE, deep-sup, 6×3)")

# Load the checkpoint from  40-epoch run
ckpt = torch.load(CKPT_DIR /"best_model.pt", map_location='cpu') 

# Strip _orig_mod. prefix from torch.compile()
clean_state = {k.replace('_orig_mod.', ''): v for k, v in ckpt["model"].items()}
missing, unexpected = original_trm.load_state_dict(clean_state, strict=False)
if missing:     print(f"  Missing keys    : {len(missing)}")
if unexpected:  print(f"  Unexpected keys : {len(unexpected)}")

ema_orig = EMA(original_trm)
ema_orig.shadow = {
    k.replace('_orig_mod.', ''): v.to(DEVICE)
    for k, v in ckpt["ema_shadow"].items()
}
ema_orig.apply(original_trm)
print(f"[Cell 11] Original TRM loaded from checkpoint (epoch {ckpt.get('epoch','?')}).")


  Original TRM (3-D RoPE, deep-sup, 6×3)
  token_emb                         1,152
  pair_emb                          1,632
  proj_in                          27,744
  f_inner                         173,504
  proj_ya                          18,528
  f_update                        173,504
  head                                970
  halt_head                            97
  drop                                  0
────────────────────────────────────────────────────────────
  TOTAL                           397,131
  ✓ Within 50M budget  (0.3971M / 50.000M)

[Cell 11] Original TRM loaded from checkpoint (epoch 36).


## Load data & build train/val splits

In [ ]:
print("[Data] Loading ARC-1 training tasks…")
raw_train = load_arc_tasks(TRAIN_DIR)
augmented = build_augmented_dataset(raw_train, colour_aug=True)

val_tasks = raw_train[-40:]   # last 40, consistent with scanner
val_ids   = {t["task_id"] for t in val_tasks}
train_aug = [t for t in augmented
             if t["task_id"].split("_d")[0].split("_cp")[0] not in val_ids]

train_ds = ARCDataset(train_aug)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True,
                      collate_fn=arc_collate_fn, num_workers=2, pin_memory=True)

print(f"[Data] Train samples    : {len(train_ds)}")
print(f"[Data] Val tasks        : {len(val_tasks)}")
print(f"[Data] Batches/epoch    : {len(train_dl)}")


[Data] Loading ARC-1 training tasks…
[Dataset] 400 tasks → 3465 augmented
[Data] Train samples    : 10125
[Data] Val tasks        : 40
[Data] Batches/epoch    : 2532


## Baseline: evaluate original TRM (40-epoch checkpoint)

This is the reference point for comparing all three ablations.


In [ ]:
print("=" * 65)
print("  BASELINE — Original TRM (3-D RoPE, deep-sup, 6×3 cycles)")
print("  Loaded from 40-epoch checkpoint")
print("=" * 65)

pix_acc_orig = evaluate_pixel_accuracy(original_trm, val_tasks, DEVICE, max_tasks=40)
solved, n, em_orig = evaluate_exact_match(original_trm, val_tasks, DEVICE, max_tasks=40)

print(f"  Params        : {orig_params:,}  ({orig_params/1e6:.4f}M)")
print(f"  Pixel Acc     : {pix_acc_orig*100:.2f}%")
print(f"  Exact Match   : {solved}/{n}  ({em_orig*100:.2f}%)")
print("=" * 65)

results = {}
results["Original TRM"] = {
    "params": orig_params,
    "pixel_acc": pix_acc_orig,
    "exact_match": em_orig,
    "epochs": 40,
    "config": "3-D RoPE | deep-supervision | 6 inner × 3 outer"
}


  BASELINE — Original TRM (3-D RoPE, deep-sup, 6×3 cycles)
  Loaded from 40-epoch checkpoint
  Params        : 397,131  (0.3971M)
  Pixel Acc     : 60.21%
  Exact Match   : 0/40  (0.00%)


## ABL-1: Replace 3-D RoPE with 2-D RoPE

**What changes:**
- `RoPE3D` → `RoPE2D` inside every attention head
- 2-D RoPE encodes only `(row, col)` — pair_index dimension dropped
- `head_dim=36` must be divisible by 4 (for 2-D) instead of 6 (for 3-D) ✓
- **Trainable parameter count is identical** — both RoPE variants are parameter-free

**Expected effect:** Without positional encoding of the pair index, the model
loses the ability to distinguish which demo pair each token came from via the
rotation alone. It must rely entirely on the learned `pair_emb`. Hypothesis:
pixel accuracy drops ~3–8 pp.


In [ ]:
NUM_EPOCHS_ABL = 10    # match original for fair comparison
ACCUM_STEPS    = 8     # same effective batch size = 32
LR             = 3e-4

# Instantiate ABL-1 model 
abl1_model = TRM(
    d_model=96, n_heads=4, d_ff=256, n_layers=2,
    n_inner=6, T_cycles=3, dropout=0.1, max_pairs=16,
    rope_cls=RoPE2D,          # ← only change
    use_deep_supervision=True
).to(DEVICE)

abl1_params = count_parameters(abl1_model, "ABL-1: TRM with 2-D RoPE (row, col only)")

# Train 
opt1   = AdamW(abl1_model.parameters(), lr=LR, weight_decay=1e-2)
steps1 = NUM_EPOCHS_ABL * (len(train_dl) // ACCUM_STEPS)
sched1 = build_wsd_scheduler(opt1, steps1)
sc1    = GradScaler('cuda')
ema1   = EMA(abl1_model)

print(f"[ABL-1] Training for {NUM_EPOCHS_ABL} epochs  ({steps1} optimiser steps)…\n")

for epoch in range(1, NUM_EPOCHS_ABL + 1):
    loss = train_one_epoch(abl1_model, train_dl, opt1, sched1, ema1, sc1,
                           DEVICE, accum_steps=ACCUM_STEPS,
                           loss_fn=deep_supervision_loss)
    lr_now = sched1.get_last_lr()[0]
    print(f"  [ABL-1 Epoch {epoch:03d}/{NUM_EPOCHS_ABL}]  Loss: {loss:.4f}  LR: {lr_now:.2e}")

    if epoch % 10 == 0 or epoch == NUM_EPOCHS_ABL:
        orig_sd = copy.deepcopy(abl1_model.state_dict())
        ema1.apply(abl1_model)
        pix = evaluate_pixel_accuracy(abl1_model, val_tasks, DEVICE, max_tasks=40)
        s, n_, em = evaluate_exact_match(abl1_model, val_tasks, DEVICE, max_tasks=40)
        print(f"    [Val] Pixel Acc: {pix*100:.2f}%  Exact Match: {s}/{n_} ({em*100:.2f}%)")
        ema1.restore(abl1_model, orig_sd)

# Final evaluation (EMA weights) 
orig_sd = copy.deepcopy(abl1_model.state_dict())
ema1.apply(abl1_model)
pix_acc_abl1 = evaluate_pixel_accuracy(abl1_model, val_tasks, DEVICE, max_tasks=40)
solved1, n1, em_abl1 = evaluate_exact_match(abl1_model, val_tasks, DEVICE, max_tasks=40)
ema1.restore(abl1_model, orig_sd)

torch.save({"model": abl1_model.state_dict(), "ema_shadow": ema1.shadow},
           CKPT_DIR / "abl1_rope2d_final.pt")

print(f"\n[ABL-1 RESULT]")
print(f"  Params        : {abl1_params:,}  ({abl1_params/1e6:.4f}M)")
print(f"  Pixel Acc     : {pix_acc_abl1*100:.2f}%")
print(f"  Exact Match   : {solved1}/{n1}  ({em_abl1*100:.2f}%)")

results["ABL-1 (2-D RoPE)"] = {
    "params": abl1_params,
    "pixel_acc": pix_acc_abl1,
    "exact_match": em_abl1,
    "epochs": NUM_EPOCHS_ABL,
    "config": "2-D RoPE | deep-supervision | 6 inner × 3 outer"
}



  ABL-1: TRM with 2-D RoPE (row, col only)
  token_emb                         1,152
  pair_emb                          1,632
  proj_in                          27,744
  f_inner                         173,504
  proj_ya                          18,528
  f_update                        173,504
  head                                970
  halt_head                            97
  drop                                  0
────────────────────────────────────────────────────────────
  TOTAL                           397,131
  ✓ Within 50M budget  (0.3971M / 50.000M)

[ABL-1] Training for 10 epochs  (3160 optimiser steps)…

  [ABL-1 Epoch 001/10]  Loss: 1.7134  LR: 3.00e-04
  [ABL-1 Epoch 002/10]  Loss: 1.5948  LR: 3.00e-04
  [ABL-1 Epoch 003/10]  Loss: 1.5597  LR: 3.00e-04
  [ABL-1 Epoch 004/10]  Loss: 1.5403  LR: 3.00e-04
  [ABL-1 Epoch 005/10]  Loss: 1.5186  LR: 3.00e-04
  [ABL-1 Epoch 006/10]  Loss: 1.4985  LR: 3.00e-04
  [ABL-1 Epoch 007/10]  Loss: 1.4755  LR: 3.00e-04
  [ABL-1 Epoch 00

## ABL-2: Replace deep supervision with plain CE loss

**What changes:**
- Loss function: `deep_supervision_loss` → `ce_only_loss`
- Only the **final** cycle's logits contribute to the gradient
- Earlier reasoning cycles receive **no gradient signal**
- Architecture, RoPE, cycles: all unchanged

**Expected effect:** Without gradient flow into intermediate cycles,
the reasoning state `z` has no incentive to produce meaningful intermediate
representations. The model may converge slower and plateau lower.
Hypothesis: pixel accuracy drops ~4–10 pp.


In [ ]:
# Instantiate ABL-2 model 
abl2_model = TRM(
    d_model=96, n_heads=4, d_ff=256, n_layers=2,
    n_inner=6, T_cycles=3, dropout=0.1, max_pairs=16,
    rope_cls=RoPE3D,           # unchanged
    use_deep_supervision=False # kept for reference; ce_only_loss passed externally
).to(DEVICE)

abl2_params = count_parameters(abl2_model, "ABL-2: TRM with CE-only loss (no deep supervision)")

# Train 
opt2   = AdamW(abl2_model.parameters(), lr=LR, weight_decay=1e-2)
steps2 = NUM_EPOCHS_ABL * (len(train_dl) // ACCUM_STEPS)
sched2 = build_wsd_scheduler(opt2, steps2)
sc2    = GradScaler('cuda')
ema2   = EMA(abl2_model)

print(f"[ABL-2] Training for {NUM_EPOCHS_ABL} epochs  ({steps2} optimiser steps)…\n")

for epoch in range(1, NUM_EPOCHS_ABL + 1):
    loss = train_one_epoch(abl2_model, train_dl, opt2, sched2, ema2, sc2,
                           DEVICE, accum_steps=ACCUM_STEPS,
                           loss_fn=ce_only_loss)    # ← only change
    lr_now = sched2.get_last_lr()[0]
    print(f"  [ABL-2 Epoch {epoch:03d}/{NUM_EPOCHS_ABL}]  Loss: {loss:.4f}  LR: {lr_now:.2e}")

    if epoch % 10 == 0 or epoch == NUM_EPOCHS_ABL:
        orig_sd = copy.deepcopy(abl2_model.state_dict())
        ema2.apply(abl2_model)
        pix = evaluate_pixel_accuracy(abl2_model, val_tasks, DEVICE, max_tasks=40)
        s, n_, em = evaluate_exact_match(abl2_model, val_tasks, DEVICE, max_tasks=40)
        print(f"    [Val] Pixel Acc: {pix*100:.2f}%  Exact Match: {s}/{n_} ({em*100:.2f}%)")
        ema2.restore(abl2_model, orig_sd)

# Final evaluation 
orig_sd = copy.deepcopy(abl2_model.state_dict())
ema2.apply(abl2_model)
pix_acc_abl2 = evaluate_pixel_accuracy(abl2_model, val_tasks, DEVICE, max_tasks=40)
solved2, n2, em_abl2 = evaluate_exact_match(abl2_model, val_tasks, DEVICE, max_tasks=40)
ema2.restore(abl2_model, orig_sd)

torch.save({"model": abl2_model.state_dict(), "ema_shadow": ema2.shadow},
           CKPT_DIR / "abl2_celoss_final.pt")

print(f"\n[ABL-2 RESULT]")
print(f"  Params        : {abl2_params:,}  ({abl2_params/1e6:.4f}M)")
print(f"  Pixel Acc     : {pix_acc_abl2*100:.2f}%")
print(f"  Exact Match   : {solved2}/{n2}  ({em_abl2*100:.2f}%)")

results["ABL-2 (CE-only loss)"] = {
    "params": abl2_params,
    "pixel_acc": pix_acc_abl2,
    "exact_match": em_abl2,
    "epochs": NUM_EPOCHS_ABL,
    "config": "3-D RoPE | CE-only loss | 6 inner × 3 outer"
}



  ABL-2: TRM with CE-only loss (no deep supervision)
  token_emb                         1,152
  pair_emb                          1,632
  proj_in                          27,744
  f_inner                         173,504
  proj_ya                          18,528
  f_update                        173,504
  head                                970
  halt_head                            97
  drop                                  0
────────────────────────────────────────────────────────────
  TOTAL                           397,131
  ✓ Within 50M budget  (0.3971M / 50.000M)

[ABL-2] Training for 10 epochs  (3160 optimiser steps)…

  [ABL-2 Epoch 001/10]  Loss: 1.7393  LR: 3.00e-04
  [ABL-2 Epoch 002/10]  Loss: 1.6220  LR: 3.00e-04
  [ABL-2 Epoch 003/10]  Loss: 1.5774  LR: 3.00e-04
  [ABL-2 Epoch 004/10]  Loss: 1.5482  LR: 3.00e-04
  [ABL-2 Epoch 005/10]  Loss: 1.5294  LR: 3.00e-04
  [ABL-2 Epoch 006/10]  Loss: 1.5161  LR: 3.00e-04
  [ABL-2 Epoch 007/10]  Loss: 1.4930  LR: 3.00e-04
  [ABL-

##  ABL-3: Reduce reasoning depth to 1 inner × 1 outer cycle

**What changes:**
- `n_inner`: 6 → 1  (inner recursion steps per outer cycle)
- `T_cycles`: 3 → 1  (outer cycles)
- Total reasoning steps: 18 → 1  (18× reduction)
- **Parameter count decreases** — shared weights are the same, but
  fewer computation steps means lower effective capacity

**Expected effect:** The model has only a single pass through `f_inner`
and `f_update` — essentially a one-shot transformer with no iterative
refinement. Hypothesis: pixel accuracy drops ~10–20 pp, demonstrating
that the recursive depth is critical.


In [ ]:
# Instantiate ABL-3 model 
abl3_model = TRM(
    d_model=96, n_heads=4, d_ff=256, n_layers=2,
    n_inner=1,    # ← 6 → 1
    T_cycles=1,   # ← 3 → 1
    dropout=0.1, max_pairs=16,
    rope_cls=RoPE3D,
    use_deep_supervision=True
).to(DEVICE)

abl3_params = count_parameters(abl3_model, "ABL-3: TRM with 1 inner × 1 outer cycle")

# Train 
opt3   = AdamW(abl3_model.parameters(), lr=LR, weight_decay=1e-2)
steps3 = NUM_EPOCHS_ABL * (len(train_dl) // ACCUM_STEPS)
sched3 = build_wsd_scheduler(opt3, steps3)
sc3    = GradScaler('cuda')
ema3   = EMA(abl3_model)

print(f"[ABL-3] Training for {NUM_EPOCHS_ABL} epochs  ({steps3} optimiser steps)…")
print(f"        Reasoning depth: 1 inner × 1 outer = 1 step total\n")

for epoch in range(1, NUM_EPOCHS_ABL + 1):
    loss = train_one_epoch(abl3_model, train_dl, opt3, sched3, ema3, sc3,
                           DEVICE, accum_steps=ACCUM_STEPS,
                           loss_fn=deep_supervision_loss)
    lr_now = sched3.get_last_lr()[0]
    print(f"  [ABL-3 Epoch {epoch:03d}/{NUM_EPOCHS_ABL}]  Loss: {loss:.4f}  LR: {lr_now:.2e}")

    if epoch % 10 == 0 or epoch == NUM_EPOCHS_ABL:
        orig_sd = copy.deepcopy(abl3_model.state_dict())
        ema3.apply(abl3_model)
        pix = evaluate_pixel_accuracy(abl3_model, val_tasks, DEVICE, max_tasks=40)
        s, n_, em = evaluate_exact_match(abl3_model, val_tasks, DEVICE, max_tasks=40)
        print(f"    [Val] Pixel Acc: {pix*100:.2f}%  Exact Match: {s}/{n_} ({em*100:.2f}%)")
        ema3.restore(abl3_model, orig_sd)

# Final evaluation 
orig_sd = copy.deepcopy(abl3_model.state_dict())
ema3.apply(abl3_model)
pix_acc_abl3 = evaluate_pixel_accuracy(abl3_model, val_tasks, DEVICE, max_tasks=40)
solved3, n3, em_abl3 = evaluate_exact_match(abl3_model, val_tasks, DEVICE, max_tasks=40)
ema3.restore(abl3_model, orig_sd)

torch.save({"model": abl3_model.state_dict(), "ema_shadow": ema3.shadow},
           CKPT_DIR / "abl3_1x1cycles_final.pt")

print(f"\n[ABL-3 RESULT]")
print(f"  Params        : {abl3_params:,}  ({abl3_params/1e6:.4f}M)")
print(f"  Pixel Acc     : {pix_acc_abl3*100:.2f}%")
print(f"  Exact Match   : {solved3}/{n3}  ({em_abl3*100:.2f}%)")

results["ABL-3 (1X1 cycles)"] = {
    "params": abl3_params,
    "pixel_acc": pix_acc_abl3,
    "exact_match": em_abl3,
    "epochs": NUM_EPOCHS_ABL,
    "config": "3-D RoPE | deep-supervision | 1 inner X 1 outer"
}



  ABL-3: TRM with 1 inner × 1 outer cycle
  token_emb                         1,152
  pair_emb                          1,632
  proj_in                          27,744
  f_inner                         173,504
  proj_ya                          18,528
  f_update                        173,504
  head                                970
  halt_head                            97
  drop                                  0
────────────────────────────────────────────────────────────
  TOTAL                           397,131
  ✓ Within 50M budget  (0.3971M / 50.000M)

[ABL-3] Training for 10 epochs  (3160 optimiser steps)…
        Reasoning depth: 1 inner × 1 outer = 1 step total

  [ABL-3 Epoch 001/10]  Loss: 1.6825  LR: 3.00e-04
  [ABL-3 Epoch 002/10]  Loss: 1.5431  LR: 3.00e-04
  [ABL-3 Epoch 003/10]  Loss: 1.5073  LR: 3.00e-04
  [ABL-3 Epoch 004/10]  Loss: 1.4743  LR: 3.00e-04
  [ABL-3 Epoch 005/10]  Loss: 1.4570  LR: 3.00e-04
  [ABL-3 Epoch 006/10]  Loss: 1.4365  LR: 3.00e-04
  [ABL-3 Ep